# 实验 6-1：Softmax 分类 (Softmax Classification)

## 导入库 (Imports)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
# 为了结果可复现
torch.manual_seed(1)

## Softmax 函数

使用 Softmax 将数字转换为概率。

$$ P(class=i) = \frac{e^i}{\sum e^i} $$

In [3]:
z = torch.FloatTensor([1, 2, 3])

PyTorch 提供了一个 `softmax` 函数。

In [5]:
hypothesis = F.softmax(z, dim=0)
print(hypothesis)

tensor([0.0900, 0.2447, 0.6652])


因为它们是概率，所以它们的和应该为 1。让我们进行一下合理性检查。

In [6]:
hypothesis.sum()

tensor(1.)

## 交叉熵损失 (Cross Entropy Loss) [低阶实现]

对于多类别分类问题，我们使用交叉熵损失函数。

$$ L = \frac{1}{N} \sum - y \log(\hat{y}) $$

其中 $\hat{y}$ 是预测概率，$y$ 是正确标签的概率（0 或 1）。

In [7]:
z = torch.rand(3, 5, requires_grad=True)
hypothesis = F.softmax(z, dim=1)
print(hypothesis)

tensor([[0.2645, 0.1639, 0.1855, 0.2585, 0.1277],
        [0.2430, 0.1624, 0.2322, 0.1930, 0.1694],
        [0.2226, 0.1986, 0.2326, 0.1594, 0.1868]], grad_fn=<SoftmaxBackward0>)


In [ ]:
y = torch.randint(5, (3,)).long()# 生成3个随机整数，每个整数在[0, 5)之间
print(y)

tensor([0, 2, 1])


In [11]:
y_one_hot = torch.zeros_like(hypothesis)
y_one_hot.scatter_(1, y.unsqueeze(1), 1)

tensor([[1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.]])

In [12]:
cost = (y_one_hot * -torch.log(hypothesis)).sum(dim=1).mean()
print(cost)

tensor(1.4689, grad_fn=<MeanBackward0>)


## 使用 `torch.nn.functional` 实现交叉熵损失

PyTorch 提供了 `F.log_softmax()` 函数。

In [13]:
# 低阶用法
torch.log(F.softmax(z, dim=1))

tensor([[-1.3301, -1.8084, -1.6846, -1.3530, -2.0584],
        [-1.4147, -1.8174, -1.4602, -1.6450, -1.7758],
        [-1.5025, -1.6165, -1.4586, -1.8360, -1.6776]], grad_fn=<LogBackward0>)

In [14]:
# 高阶用法
F.log_softmax(z, dim=1)

tensor([[-1.3301, -1.8084, -1.6846, -1.3530, -2.0584],
        [-1.4147, -1.8174, -1.4602, -1.6450, -1.7758],
        [-1.5025, -1.6165, -1.4586, -1.8360, -1.6776]],
       grad_fn=<LogSoftmaxBackward0>)

PyTorch 还提供了 `F.nll_loss()` 函数，用于计算负对数似然损失 (Negative Log Likelihood Loss)。

In [15]:
# 低阶用法
(y_one_hot * -torch.log(F.softmax(z, dim=1))).sum(dim=1).mean()

tensor(1.4689, grad_fn=<MeanBackward0>)

In [16]:
# 高阶用法
F.nll_loss(F.log_softmax(z, dim=1), y)

tensor(1.4689, grad_fn=<NllLossBackward0>)

PyTorch 还提供了 `F.cross_entropy`，它结合了 `F.log_softmax()` 和 `F.nll_loss()`。

In [17]:
F.cross_entropy(z, y)

tensor(1.4689, grad_fn=<NllLossBackward0>)

## 使用低阶交叉熵损失进行训练

In [18]:
x_train = [[1, 2, 1, 1],
           [2, 1, 3, 2],
           [3, 1, 3, 4],
           [4, 1, 5, 5],
           [1, 7, 5, 5],
           [1, 2, 5, 6],
           [1, 6, 6, 6],
           [1, 7, 7, 7]]
y_train = [2, 2, 2, 1, 1, 1, 0, 0]
x_train = torch.FloatTensor(x_train)
y_train = torch.LongTensor(y_train)

In [19]:
# 模型初始化
W = torch.zeros((4, 3), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# 优化器设置
optimizer = optim.SGD([W, b], lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # 计算 Cost (1)
    hypothesis = F.softmax(x_train.matmul(W) + b, dim=1) # or .mm or @
    y_one_hot = torch.zeros_like(hypothesis)
    y_one_hot.scatter_(1, y_train.unsqueeze(1), 1)
    cost = (y_one_hot * -torch.log(F.softmax(hypothesis, dim=1))).sum(dim=1).mean()

    # 使用 Cost 更新 H(x)
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 每 100 次迭代输出一次日志
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/1000 Cost: 1.098612
Epoch  100/1000 Cost: 0.901535
Epoch  200/1000 Cost: 0.839114
Epoch  300/1000 Cost: 0.807826
Epoch  400/1000 Cost: 0.788472
Epoch  500/1000 Cost: 0.774822
Epoch  600/1000 Cost: 0.764449
Epoch  700/1000 Cost: 0.756191
Epoch  800/1000 Cost: 0.749398
Epoch  900/1000 Cost: 0.743671
Epoch 1000/1000 Cost: 0.738749


## 使用 `F.cross_entropy` 进行训练

In [17]:
# 模型初始化
W = torch.zeros((4, 3), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# 优化器设置
optimizer = optim.SGD([W, b], lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # 计算 Cost (2)
    z = x_train.matmul(W) + b # or .mm or @
    cost = F.cross_entropy(z, y_train)

    # 使用 Cost 更新 H(x)
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 每 100 次迭代输出一次日志
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/1000 Cost: 1.098612
Epoch  100/1000 Cost: 0.761050
Epoch  200/1000 Cost: 0.689991
Epoch  300/1000 Cost: 0.643229
Epoch  400/1000 Cost: 0.604117
Epoch  500/1000 Cost: 0.568255
Epoch  600/1000 Cost: 0.533922
Epoch  700/1000 Cost: 0.500291
Epoch  800/1000 Cost: 0.466908
Epoch  900/1000 Cost: 0.433507
Epoch 1000/1000 Cost: 0.399962


## 使用 `nn.Module` 的高阶实现

In [20]:
class SoftmaxClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 3) # 输出为 3！

    def forward(self, x):
        return self.linear(x)

In [23]:
model = SoftmaxClassifierModel()

让我们尝试另一个新的数据集。

In [24]:
# 优化器设置
optimizer = optim.SGD(model.parameters(), lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # 计算 H(x)
    prediction = model(x_train)

    # 计算 Cost
    cost = F.cross_entropy(prediction, y_train)

    # 使用 Cost 更新 H(x)
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()
    
    # 每 100 次迭代输出一次日志
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/1000 Cost: 3.586499
Epoch  100/1000 Cost: 0.650367
Epoch  200/1000 Cost: 0.566842
Epoch  300/1000 Cost: 0.512495
Epoch  400/1000 Cost: 0.468165
Epoch  500/1000 Cost: 0.428759
Epoch  600/1000 Cost: 0.391982
Epoch  700/1000 Cost: 0.356357
Epoch  800/1000 Cost: 0.320727
Epoch  900/1000 Cost: 0.284398
Epoch 1000/1000 Cost: 0.250610
